# Train GNN trên Google Colab — Drama Intelligence

**Mục tiêu:** Extract PhoBERT [CLS] embedding cho ~26k comment (Edu + Show), concat với TF-IDF + structural, rồi train 2 GraphSAGE classifier riêng.

**Trước khi chạy:**
1. Upload thư mục `ml/gnn/` (code) và `data/gnn/` (graphs.pkl + tfidf.npz đã sinh sẵn ở local) lên Drive `/MyDrive/NCKH_hong_drama/`.
2. Chọn Runtime > Change runtime type > GPU (T4 free OK).
3. Run all cells theo thứ tự.

## 1. Mount Drive + cài deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers sentencepiece
!pip install -q torch-geometric

In [ ]:
import os, shutil
PROJECT = '/content/drive/MyDrive/NCKH_hong_drama'
os.chdir(PROJECT)
print(os.listdir('.'))

## 2. Verify GPU + data đã upload

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
for f in ['data/gnn/graphs_education.pkl', 'data/gnn/graphs_showbiz.pkl',
          'data/gnn/tfidf_education.npz', 'data/gnn/tfidf_showbiz.npz']:
    print(f, '->', os.path.exists(f))

## 3. Extract PhoBERT [CLS] embedding (~10 phút trên T4)

In [ ]:
import sys; sys.path.insert(0, 'ml/gnn')
!python ml/gnn/extract_phobert_features.py --domain all --batch-size 64

## 4. Concat features (PhoBERT 768 + TF-IDF 300 + structural 3 = 1071d)

In [ ]:
!python ml/gnn/concat_node_features.py --domain all

## 5. Train GNN — Education (4 lớp)

In [ ]:
!cd ml/gnn && python train_gnn_node_classification.py --domain education --epochs 300 --patience 50

## 6. Train GNN — Showbiz (5 lớp, có class weight do imbalance 94% trung lập)

In [ ]:
!cd ml/gnn && python train_gnn_node_classification.py --domain showbiz --epochs 300 --patience 50

## 7. Evaluate vs LLM

In [ ]:
!cd ml/gnn && python evaluate_gnn_vs_llm.py --domain all

In [ ]:
import json
for d in ['education', 'showbiz']:
    print('===', d)
    print(json.dumps(json.load(open(f'data/gnn/eval_gnn_vs_llm_{d}.json', encoding='utf-8')), ensure_ascii=False, indent=2))
print('=== summary ==='); print(open('data/gnn/eval_summary.md', encoding='utf-8').read())

## 8. Sau khi xong

Output đã lưu trên Drive ở `data/gnn/`:
- `phobert_{domain}.npy` — PhoBERT embedding
- `node_features_{domain}.npy` — hybrid 1071d
- `checkpoints/gnn_{domain}.pt` — model trained
- `metrics_{domain}.json` — F1, confusion matrix
- `predictions_{domain}.json` — prediction toàn bộ node
- `eval_gnn_vs_llm_{domain}.json` + `eval_summary.md` — so sánh vs LLM

Báo cho controller biết để viết phần GNN vào Word.